In [1]:
import os, re, json, chromadb, uuid
from dotenv import load_dotenv

In [2]:
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in environment variables. Check your .env file.")

pine_api_key = os.getenv("PINNECONE_API_KEY")
if not pine_api_key:
    raise ValueError("PINNECONE_API_KEY not found in environment variables. Check your .env file.")

In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader

def load_documents():
    folder_path = '../documents'
    all_docs = []
    num_docs = 0

    if not os.path.exists(folder_path):
        print(f"Error: Folder {folder_path} does not exist.")
        return []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            num_docs += 1
            all_docs.extend(doc)
            
    print(f"Successfully loaded {num_docs}.")
    print(f"Successfully loaded {len(all_docs)} total pages from PDFs.")
    return all_docs # Don't forget to return the list!

# Usage
documents = load_documents()


Successfully loaded 5.
Successfully loaded 36 total pages from PDFs.


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs_in_chunks(documnets):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap = 500
    )
    
    chunked_docs = text_splitter.split_documents(documnets)
    print(f"Total Chunks : {len(chunked_docs)}")
    return chunked_docs

chunked_docs = split_docs_in_chunks(documents)  
    

Total Chunks : 185


In [5]:
# for doc in chunked_docs:
#     print(doc.metadata.get("source", "unknown"))
#     print(doc.metadata.get("page_label", 0))

In [6]:
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        print("Loading Model....", self.model_name)
        self.model=SentenceTransformer(self.model_name)
        print("Embeddings Dimensions : ", self.model.get_sentence_embedding_dimension())
        
    def generate_embedding(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("Embedding Shape : ", embeddings.shape)
        return embeddings     

In [7]:
embeddings_manager=EmbeddingManager()

Loading Model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings Dimensions :  384


In [8]:
import uuid
from pinecone import Pinecone, ServerlessSpec
import time

class PineconeManager:
    def __init__(self, api_key: str, index_name: str, dimension: int):
        self.pc = Pinecone(api_key=api_key) 
        self.index_name = index_name
        self.dimension = dimension
        self._initialize_index()
        self.index = self.pc.Index(index_name)
        
    def _initialize_index(self):
        existing_indexes = [idx.name for idx in self.pc.list_indexes()]
        if self.index_name not in existing_indexes:
            print(f"Creating index: {self.index_name}...")
            self.pc.create_index(
                name=self.index_name,
                dimension=self.dimension,
                metric="cosine",
                spec=ServerlessSpec(cloud="aws", region="us-east-1")
            )
            while not self.pc.describe_index(self.index_name).status['ready']:
                time.sleep(1)
        else:
            print(f"Connecting to existing index: {self.index_name}")
            
    def upsert_embeddings(self, chunks, embeddings_manager, namespace="pdf-documents"):
        vectors_to_upsert = [] 
        for doc in chunks:
            # Generate embedding
            vectors = embeddings_manager.generate_embedding(doc.page_content).tolist()
            vectors_id = str(uuid.uuid4())
            vector_metadata = {
                "text": doc.page_content,
                "source": doc.metadata.get("source", "unknown"),
                "page": doc.metadata.get("page_label", 0)
            }
            vectors_to_upsert.append((vectors_id, vectors, vector_metadata))
            
            # Batch upsert every 100 vectors
            if len(vectors_to_upsert) >= 100:
                self.index.upsert(vectors=vectors_to_upsert, namespace=namespace)
                vectors_to_upsert = []
                
        if len(vectors_to_upsert) > 0:
            self.index.upsert(vectors=vectors_to_upsert, namespace=namespace)
            
        print(f"Successfully uploaded {len(chunks)} chunks to Pinecone.")
        
    def retrieve(self, query_text: str, embedding_manager, top_k: int = 5, namespace: str = "pdf-documents"):
        query_vector = embedding_manager.model.encode(query_text).tolist()
        
        results = self.index.query(
            vector=query_vector,
            top_k=top_k,
            include_metadata=True,
            namespace="pdf-documents"
        )

        return results


In [9]:
pc_manager=PineconeManager(api_key=pine_api_key, index_name="pdf-search-index", dimension=384)

Connecting to existing index: pdf-search-index


In [10]:
# pc_manager.upsert_embeddings(chunked_docs, embeddings_manager)

In [12]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
def generate_output(api_key, query, rag_retriever, embeddings_manager, top_k=5):
    # 1. Retrieve the top_k relevant chunks from Pinecone
    results = rag_retriever.retrieve(query, embeddings_manager, top_k=top_k)
    
    # 2. Extract and join the text from the metadata
    # We join them with newlines to create a cohesive context block
    context_chunks = [match['metadata']['text'] for match in results['matches']]
    context_text = "\n\n---\n\n".join(context_chunks)
    
    # 3. Initialize the Groq LLM
    llm = ChatGroq(
        temperature=0, 
        groq_api_key=api_key, 
        model_name="llama-3.3-70b-versatile"
    )
    
    # 4. Construct the prompt
    system_prompt = (
        "You are a helpful assistant. Use the following pieces of retrieved context "
        "to answer the user's question. If the answer is not in the context, "
        "just say that you don't know. Don't try to make up an answer."
    )
    
    user_content = f"Context:\n{context_text}\n\nQuestion: {query}"
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_content),
    ]
    
    # 5. Generate response
    response = llm.invoke(messages)
    
    return response.content

# --- Usage ---
groq_api_key = groq_api_key
query = 'Give a list of all general exclusions'

# Run the full pipeline
output = generate_output(groq_api_key, query, pc_manager, embeddings_manager)
print(output)

2 7
2 8
3 0
3 1
3 2
3 3
3 4
3 5
3 6
3 7
3 8
4 0
4 1
4 2
4 3
4 4
4 5
4 6
4 7
4 8
4 9
5 0
5 1
5 2
5 3
5 4
5 5
5 6
5 7
5 8
5 9
6 0
6 1
6 2
6 3
6 4
6 5
6 6
6 7
6 8
6 9
7 0
7 1
7 2
7 3
7 4
7 5
7 6
7 7
7 8
7 9
8 0
8 1
8 2
8 3
8 4
8 5
8 6
8 7
8 8
8 9
9 0
9 1
9 2
9 3
9 4
9 5
9 6
9 7
9 8
9 9
1 0 0
1 0 1
1 0 2
1 0 3
1 0 4
1 0 5
1 0 6
1 0 7
1 0 8
1 0 9
1 1 0
1 1 1
1 1 2
1 1 3
1 1 4
1 1 5
1 1 6
1 1 7
1 1 8
1 1 9
1 2 0
1 2 1
1 2 2
1 2 3
1 2 4
1 2 5
1 2 6
1 2 7
1 2 8
1 2 9
1 3 0
1 3 1
1 3 2
1 3 3
1 3 4
1 3 5
1 3 6
1 3 7
1 3 8
1 3 9
1 4 0
1 4 1
1 4 2
1 4 3
1 4 4
1 4 5
1 4 6
1 4 7
1 4 8
1 4 9
1 5 0
1 5 1
1 5 2
1 5 3
1 5 4
1 5 5
1 5 6
1 5 7
1 5 8
1 5 9
1 6 0
1 6 1
1 6 2
1 6 3
1 6 4
1 6 5
1 6 6
1 6 7
1 6 8
1 6 9
1 7 0
1 7 1
1 7 2
1 7 3
1 7 4
1 7 5
1 7 6
1 7 7
1 7 8
1 7 9
1 8 0
1 8 1
1 8 2
1 8 3
1 8 4
1 8 5
1 8 6
1 8 7
1 8 8
1 8 9
1 9 0
1 9 1
1 9 2
1 9 3
1 9 4
1 9 5
1 9 6
1 9 7
1 9 8
1 9 9
2 0 0
2 0 1
2 0 2
2 0 3
2 0 4
2 0 5
2 0 6
2 0 7
2 0 8
2 0 9
2 1 0
2 1 1
2 1 2
2 1 3
2 1 4
2 1 5
2 1 6
2 1 7
2 1 8
2 